In [1]:
%load_ext autoreload
%autoreload 2

%matplotlib inline

In [2]:
from pathlib import Path

import torch

import numpy as np
import pandas as pd

from captum.attr import LRP
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate

from magneton.data import SupervisedDownstreamTaskDataModule
from magneton.data.evaluations import DeepFriModule, TASK_TO_TYPE
from magneton.utils import get_data_dir
from magneton.models.evaluation_classifier import EvaluationClassifier

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
ckpt_path = "/weka/scratch/weka/kellislab/rcalef/projects/magneton/experiments/downstream_evals/esmc_300m_abc_cutoff25/combined_evals_no_ft/GO:MF/final_model.ckpt"

model = EvaluationClassifier.load_from_checkpoint(ckpt_path)

INFO:magneton.models.substructure_classifier:head model: ModuleDict(
  (Active_site): Sequential(
    (0): Linear(in_features=960, out_features=960, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=960, out_features=114, bias=True)
  )
  (Binding_site): Sequential(
    (0): Linear(in_features=960, out_features=960, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=960, out_features=72, bias=True)
  )
  (Conserved_site): Sequential(
    (0): Linear(in_features=960, out_features=960, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=960, out_features=573, bias=True)
  )
)
INFO:magneton.models.evaluation_classifier:head model: ProteinClassificationHead(
  (mlp): Sequential(
    (0): Dropout(p=0.1, inplace=False)
    (1): Linear(in_features=960, out_features=960, bias=True)
    (2): Tanh()
    (3): Dropout(p=0.1, inplace=False)
    (4): Linear(in_features=960

In [7]:
config_dir = "/home/rcalef/sandbox/repos/magneton/magneton/configs"
with initialize_config_dir(config_dir=config_dir, version_base=None):
      cfg = compose(config_name="config", overrides=[
            "+evaluate=deepfri",
            "output_dir='/tmp/'",
            "evaluate.model_checkpoint='/tmp'",
            "data.struct_template='/weka/scratch/weka/kellislab/rcalef/data/pdb_alphafolddb/AF-%s-F1-model_v4.pdb'",
            "data.batch_size=1",
      ])
config = instantiate(cfg)
config

PipelineConfig(_target_='magneton.config.PipelineConfig', seed=42, stage='train', output_dir='/tmp/', run_id='my_new_run', data=DataConfig(_target_='magneton.config.DataConfig', data_dir='/weka/scratch/weka/kellislab/rcalef/data/magneton-data/interpro_103.0/swissprot_subset', compression='gz', prefix='swissprot.with_ss', splits='/weka/scratch/weka/kellislab/rcalef/data/magneton-data/interpro_103.0/dataset_splits/seq_splits.tsv', batch_size=1, fasta_path='/weka/scratch/weka/kellislab/rcalef/data/magneton-data/sequences/uniprot_sprot.fasta.gz', labels_path='/weka/scratch/weka/kellislab/rcalef/data/magneton-data/interpro_103.0/labels/selected_subset', struct_template='/weka/scratch/weka/kellislab/rcalef/data/pdb_alphafolddb/AF-%s-F1-model_v4.pdb', substruct_types=['Domain'], collapse_labels=False, num_loader_workers=32, model_specific_params={}), base_model=BaseModelConfig(_target_='magneton.config.BaseModelConfig', model='esmc', model_params={'model_size': '600m', 'weights_path': '/weka/

In [8]:
module = SupervisedDownstreamTaskDataModule(
    data_config=config.data,
    task="GO:MF",
    data_dir=config.evaluate.data_dir,
    model_type="esmc",
)
deepfri_module = DeepFriModule(
    task="MF",
    data_dir=module.data_dir,
    struct_template=config.data.struct_template,
)

In [178]:
mf_df = deepfri_module.get_dataset("test").df

INFO:magneton.data.evaluations.deepfri_dataset:found PDB-UniProt mapping at: /weka/scratch/weka/kellislab/rcalef/data/magneton-data/evaluations/GeneOntology/MF.pdb_to_uniprot.tsv
INFO:magneton.data.evaluations.deepfri_dataset:downloading 1557 / 30069 missing files
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has alre

In [179]:
mf_df.head()

,protein_id,MF,labels,uniprot_id,structure_path,seq
0,11AS-A,"GO:0097367,GO:0042802,GO:0016879,GO:0032559,GO...","[tensor(466), tensor(393), tensor(288), tensor...",P00963,/weka/scratch/weka/kellislab/rcalef/data/pdb_a...,MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQD...
5,18GS-A,"GO:0030234,GO:0019899,GO:0019901,GO:0033218,GO...","[tensor(350), tensor(329), tensor(331), tensor...",P09211,/weka/scratch/weka/kellislab/rcalef/data/pdb_a...,MPPYTVVYFPVRGRCAALRMLLADQGQSWKEEVVTVETWQEGSLKA...
17,1A0P-A,"GO:0140097,GO:0003677","[tensor(476), tensor(10)]",P0A8P8,/weka/scratch/weka/kellislab/rcalef/data/pdb_a...,MKQDLARIEQFLDALWLEKNLAENTLNAYRRDLSMMVEWLHHRGLT...
22,1A22-A,"GO:0005126,GO:0051427,GO:0030545,GO:0005102,GO...","[tensor(100), tensor(434), tensor(354), tensor...",P01241,/weka/scratch/weka/kellislab/rcalef/data/pdb_a...,MATGSRTSLLLAFGLLCLPWLQEGSAFPTIPLSRLFDNAMLRAHRL...
34,1A4E-A,"GO:0016209,GO:0004601,GO:0046906,GO:0016684,GO...","[tensor(196), tensor(72), tensor(410), tensor(...",P15202,/weka/scratch/weka/kellislab/rcalef/data/pdb_a...,MSKLGQEKNEVNYSDVREDRVVTNSTGNPINEPFVTQRIGEHGPLL...


In [9]:
loader = module.val_dataloader()

INFO:magneton.data.evaluations.deepfri_dataset:found PDB-UniProt mapping at: /weka/scratch/weka/kellislab/rcalef/data/magneton-data/evaluations/GeneOntology/MF.pdb_to_uniprot.tsv
INFO:magneton.data.evaluations.deepfri_dataset:downloading 1557 / 30069 missing files
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1557/1557 [00:09<00:00, 155.71it/s]
INFO:magneton.data.evaluations.utils:succesfully downloaded 0  / 1557 files
INFO:magneton.data.evaluations.utils:FASTA cache found at /weka/scratch/weka/kellislab/rcalef/data/magneton-data/evaluations/GeneOntology/MF.seqs_from_pdbs.fa
INFO:magneton.data.data_modules:remaining proteins after length filter: 2484 / 2495


In [10]:
itor = iter(loader)
example = next(itor)

In [11]:
example

ESMCBatch(protein_ids=['1A0A-A'], lengths=[312], seqs=None, substructures=None, structure_list=None, labels=tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
 

In [12]:
model = model.to(device)
example = example.to(device)

In [13]:
with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    out = model(example)
out.shape

torch.Size([1, 489])

In [14]:
torch.nonzero(example.labels)

tensor([[  0, 415]], device='cuda:0')

In [16]:
from captum.attr import LayerLRP

In [18]:
model.base_model.model.transformer

TransformerStack(
  (blocks): ModuleList(
    (0-29): 30 x UnifiedTransformerBlock(
      (attn): FlashMultiHeadAttention(
        (layernorm_qkv): Sequential(
          (0): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=960, out_features=2880, bias=False)
        )
        (out_proj): Linear(in_features=960, out_features=960, bias=False)
        (q_ln): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
        (k_ln): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
        (rotary): TritonRotaryEmbedding()
      )
      (ffn): Sequential(
        (0): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
        (1): Linear(in_features=960, out_features=5120, bias=False)
        (2): SwiGLU()
        (3): Linear(in_features=2560, out_features=960, bias=False)
      )
    )
  )
  (norm): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
)

In [19]:
lrp = LayerLRP(model=model, layer=model.base_model.model.transformer)

In [24]:
attribution = lrp.attribute(example, target=415)

TypeError: Module of type <class 'torch.nn.modules.sparse.Embedding'> has no rule defined and nodefault rule exists for this module type. Please, set a ruleexplicitly for this module and assure that it is appropriatefor this type of layer.

In [21]:
from captum.attr import Saliency

In [29]:
saliency = Saliency(model.head.mlp)

In [143]:
import esm
import torch.nn as nn

from esm.models.esmc import ESMC, ESMCOutput
from magneton.data.model_specific.esmc import ESMCBatch

from flash_attn.bert_padding import pad_input, unpad_input  # type:ignore

is_flash_attn_available = True

In [171]:
from captum.attr._utils.lrp_rules import EpsilonRule

class ESMCForInterp(nn.Module):
    def __init__(
        self,
        oth_model: EvaluationClassifier,
    ):
        super().__init__()
        self.esmc_transformer = oth_model.base_model.model.transformer
        self.esmc_sequence_head = oth_model.base_model.model.sequence_head
        self._use_flash_attn = oth_model.base_model.model._use_flash_attn
        self.tokenizer = oth_model.base_model.model.tokenizer

        self.head = oth_model.head
        self.rep_layer = oth_model.base_model.rep_layer

        for module in self.modules():
        #print(type(module))
            if type(module) in [
                torch.nn.modules.normalization.LayerNorm,
                esm.layers.rotary.TritonRotaryEmbedding,
                esm.layers.blocks.SwiGLU,
                torch.nn.modules.activation.GELU,
            ]:
                module.rule = EpsilonRule()

    def forward(
        self,
        embeds: torch.Tensor,
    ) -> torch.Tensor:
        B, L = embeds.shape[:2]
        sequence_id = torch.ones((B, L), device=embeds.device).bool()

        # If sequence_id looks like a mask.
        if self._use_flash_attn:
            assert (
                sequence_id.dtype == torch.bool
            ), "sequence_id must be a boolean mask if Flash Attention is used"
            assert sequence_id.shape == (B, L)
            assert unpad_input is not None
            embeds, indices, *_ = unpad_input(  # type: ignore
                embeds, sequence_id
            )
        else:
            indices = None


        x, _, hiddens = self.esmc_transformer(embeds, sequence_id=sequence_id)


        if self._use_flash_attn:
            assert indices is not None
            assert pad_input is not None
            x = pad_input(x, indices, B, L)  # Back to [B, L, D]
            hiddens = [
                # Back to [[B, L, D], ...]
                pad_input(h, indices, B, L)
                for h in hiddens
            ]

        # Stack hidden states into a [n_layers, B, L, D] matrix.
        hiddens = torch.stack(hiddens, dim=0)  # type: ignore

        sequence_logits = self.esmc_sequence_head(x)
        output = ESMCOutput(
            sequence_logits=sequence_logits, embeddings=x, hidden_states=hiddens
        )
        residue_embeds = self.esmc_transformer.norm(output.hidden_states[self.rep_layer])
        print(residue_embeds[:5, :5])
        protein_embeds = residue_embeds[:, 0, :]
        print(protein_embeds[:5, :5])
        return self.head.mlp(protein_embeds)


In [172]:
interp_model = ESMCForInterp(
    model,
)

In [173]:
model.eval()
_ = interp_model.eval()

In [174]:
lrp = LRP(interp_model)

In [175]:
with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    embeds = model.base_model.model.embed(example.tokenized_seq)
    attribution = lrp.attribute(embeds, target=415)
attribution.shape

AttributeError: 'int' object has no attribute 'data'

In [156]:
with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    out = model(example)
print(out.shape)
out[:5, :5]

tensor([[[ 0.0005, -0.0079, -0.0016,  ...,  0.0039, -0.0065, -0.0069],
         [-0.0002,  0.0191,  0.0228,  ...,  0.0246, -0.0008,  0.0053],
         [-0.0698, -0.0145,  0.0163,  ...,  0.0225, -0.0365,  0.0062],
         [-0.0029,  0.0181,  0.0016,  ...,  0.0074, -0.0011,  0.0009],
         [-0.0143,  0.0032, -0.0288,  ...,  0.0063,  0.0051, -0.0142]]],
       device='cuda:0')
tensor([[ 0.0005, -0.0079, -0.0016, -0.0034, -0.0070]], device='cuda:0')
torch.Size([1, 489])


tensor([[-11.1875,  -6.7812,  -5.8125, -10.3750,  -7.9375]], device='cuda:0',
       dtype=torch.bfloat16, grad_fn=<SliceBackward0>)

In [157]:
with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    embeds = model.base_model.model.embed(example.tokenized_seq)
    out = interp_model(embeds)
print(out.shape)
out[:5, :5]

tensor([[[ 0.0005, -0.0079, -0.0016,  ...,  0.0039, -0.0065, -0.0069],
         [-0.0002,  0.0191,  0.0228,  ...,  0.0246, -0.0008,  0.0053],
         [-0.0698, -0.0145,  0.0163,  ...,  0.0225, -0.0365,  0.0062],
         [-0.0029,  0.0181,  0.0016,  ...,  0.0074, -0.0011,  0.0009],
         [-0.0143,  0.0032, -0.0288,  ...,  0.0063,  0.0051, -0.0142]]],
       device='cuda:0')
tensor([[ 0.0005, -0.0079, -0.0016, -0.0034, -0.0070]], device='cuda:0')
torch.Size([1, 489])


tensor([[-11.1875,  -6.7812,  -5.8125, -10.3750,  -7.9375]], device='cuda:0',
       dtype=torch.bfloat16, grad_fn=<SliceBackward0>)

In [84]:
saliency = Saliency(interp_model)

In [88]:
with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    embeds = interp_model.get_embeds(example)
    attribution = saliency.attribute(embeds, target=415).sum(dim=-1)
attribution.shape

tensor([[[ 0.0005, -0.0079, -0.0016,  ...,  0.0039, -0.0065, -0.0069],
         [-0.0002,  0.0191,  0.0228,  ...,  0.0246, -0.0008,  0.0053],
         [-0.0698, -0.0145,  0.0163,  ...,  0.0225, -0.0365,  0.0062],
         [-0.0029,  0.0181,  0.0016,  ...,  0.0074, -0.0011,  0.0009],
         [-0.0143,  0.0032, -0.0288,  ...,  0.0063,  0.0051, -0.0142]]],
       device='cuda:0', grad_fn=<SliceBackward0>)
tensor([[ 0.0005, -0.0079, -0.0016, -0.0034, -0.0070]], device='cuda:0',
       grad_fn=<SliceBackward0>)


torch.Size([1, 314])

In [150]:
attributions = attribution.detach().cpu().squeeze().numpy()
attributions = (attributions - np.min(attributions)) / (np.max(attributions) - np.min(attributions))

In [119]:
from IPython.display import display, HTML
#from IPython.core.display import display, HTML


In [122]:
words = model.base_model.model.tokenizer.batch_decode(example.tokenized_seq)[0].split()

html_output = ""
for word, attr in zip(words, attributions):
    if word == "<cls>":
        word = "[CLS]"
    if word == "<eos>":
        word = "[EOS]"

    # Using a simple red color scale for positive attributions
    # You would typically handle positive/negative attributions differently
    color_intensity = int(attr * 255)
    html_output += f'<span style="background-color: rgba(255, 0, 0, {attr}); padding: 2px;">{word}</span> '

# Display in a Jupyter notebook
display(HTML(html_output))

In [176]:
example

ESMCBatch(protein_ids=['1A0A-A'], lengths=[312], seqs=None, substructures=None, structure_list=None, labels=tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
 

In [126]:
next(interp_model.esmc_transformer.modules())

TransformerStack(
  (blocks): ModuleList(
    (0-29): 30 x UnifiedTransformerBlock(
      (attn): FlashMultiHeadAttention(
        (layernorm_qkv): Sequential(
          (0): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=960, out_features=2880, bias=False)
        )
        (out_proj): Linear(in_features=960, out_features=960, bias=False)
        (q_ln): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
        (k_ln): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
        (rotary): TritonRotaryEmbedding()
      )
      (ffn): Sequential(
        (0): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
        (1): Linear(in_features=960, out_features=5120, bias=False)
        (2): SwiGLU()
        (3): Linear(in_features=2560, out_features=960, bias=False)
      )
    )
  )
  (norm): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
)

In [ ]:
for